## Anomaly Detection - Credit Card Fraud Analysis

* Introduction
   - Anomaly detection identifies unusual patterns that do not follows the expected behaviour, called outliers. In this notebook, we use anomaly detection to credit card fraud using simple statistics and machine learning (Isolation Forest, LOF) approaches.

* Problem statement
   - Using the same Kaggle dataset on Credit Card Fraud Detection, model past credit card transactions to identify anomalies that correspond to fraudulent transaction. Aim for higher fraudulent transactions while minimising false positive.

* Dataset
   - Data available at [Kaggle - Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

### Import libraries

In [ ]:
# For EDA
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

# Anomaly detection models
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

# Evaluation metrics
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42

### Data loading

In [ ]:
import os
os.system('kaggle datasets download -d mlg-ulb/creditcardfraud -p /tmp --unzip')

In [ ]:
# Load data from tmp folder - dataset has been downloaded here from CC Fraud Detection project
df = pd.read_csv("/tmp/creditcard.csv")
df.head()

In [ ]:
# Inspect data structure
df.shape
#df.info

In [ ]:
#df.describe()

In [ ]:
# Check missing values
df.isnull().sum()

### Check 'Class' distribution (0 = Valid, 1 = fraud)

In [ ]:
# Check class distribution to determine fraud and valid transactions in the dataset
# Get the count of each class
fraud_count = df["Class"].value_counts()
print(f"Valid transactions: {fraud_count[0]}")
print(f"Fraudulent transactions: {fraud_count[1]}")
print(f"Fraud percentage: {fraud_count[1] / df.shape[0] * 100:.3f}%")


sns.countplot(x="Class", data=df, palette="coolwarm")
plt.title("Class Distribution: Valid vs Fraud")
plt.xticks([0, 1], ["Valid", "Fraud"])
plt.xlabel("Class")
plt.ylabel("Frequency")
plt.show()



### Check transaction amount distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[df['Class'] == 0]['Amount'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Valid Transaction Amounts')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frequency')

axes[1].hist(df[df['Class'] == 1]['Amount'], bins=50, color='crimson', edgecolor='white')
axes[1].set_title('Fraudulent Transaction Amounts')
axes[1].set_xlabel('Amount')

plt.tight_layout()
plt.show()

### Check transaction time distribution

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

sns.histplot(data = df[df['Class'] == 0], x = 'Time', bins = 250, color = 'steelblue', edgecolor = 'white', ax = axes[0])
axes[0].set_title('Valid Transactions Over Time')
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Frequency')

sns.histplot(data = df[df['Class'] == 1], x = 'Time', bins = 250, color = 'crimson', edgecolor = 'white', ax = axes[1] )
axes[1].set_title('Fraudulent Transactions Over Time')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

*Observations*
- Valid transactions show a clear cyclical pattern with two distinct peaks (~75,000s and ~150,000s) and two quieter periods across the dataset, suggesting activity follows a regular pattern over time.

- Fraudulent transactions show no distinguishing pattern, they are sparse and noisy across all time windows with occassional spikes but no particular trend.

- This suggests 'Time' has low discriminatory power for fraud detection and can be dropped during preprocessing.


### Check transaction patterns by amount over time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

sns.scatterplot(data = df[df['Class'] == 0], x = 'Time', y = 'Amount', color = 'steelblue', alpha = 0.5, s = 7, ax = axes[0])
axes[0].set_title('Valid Transactions Amount Over Time')
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Amount (USD $)')

sns.scatterplot(data = df[df['Class'] == 1], x = 'Time', y = 'Amount', color = 'crimson', alpha = 0.5, s = 10, ax = axes[1])
axes[1].set_title('Fraudulent Transactions Amount Over Time')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Amount (USD $)')

plt.tight_layout()
plt.show()

*Observations*
- As expected, the scatterplot mirrored the histograms above, valid transactions clustered densely at low amounts across all time windows with a few outliers  exceeding USD 20K $.
- Fraudulent transactions are sparse across all time windows with no discernible temporal pattern. Fraud transaction amounts are noticebaly lower than valid transactions, with max just above USD 2K $.
- Interestingly, the outliers in high-value valid transactions can be flagged as fraud using amount-based thresholding alone. This suggest limitation in simple statistics methods.
- This highlights why ML-based anomaly detection across all dataset features is necessary to establish correlations, and distinguish fraud from valid high-value transactions. 

### Summary statistics

In [ ]:
print("Summary statistics by Class:")
df.groupby("Class")["Amount"].describe().round(2)

*Observation*
- Valid transactions have a mean amount of USD 88.29, while fraudulent transactions have a higher mean of USD 122.21.
- Despite the higher mean, fraud transaction amounts have a lower maximum (USD 2,125.87) compared to valid transactions (USD 25,691.16), suggesting fraudsters avoid very large transactions.
- Standard deviation is comparable between classes (valid: USD 250.11, fraud: USD 256.68), suggesting similar spread in transaction amounts overall. However, relative to their respective means, fraud transactions show greater variability — a std of USD 256.68 against a mean of USD 122.21 versus USD 250.11 against USD 88.29 for valid transactions.

### Features correlation

In [ ]:
correlation_matrix = df.corr()
fig = plt.figure(figsize = (12, 9))
sns.heatmap(correlation_matrix, cmap = 'coolwarm', vmin = -0.8, vmax = 0.8, square = True)
plt.title('Feature Correlation Matrix')
plt.show()

*Observations*
- The diagonal confirms each feature is perfectly correlated with itself (expected).
- V1-V28 features show very little correlation with each other, confirming they are independent PCA components by design.
- Several features show notable correlation with 'Class':
   - Positive correlation (orange/reddish in Class column): V2, V4, V8, V11 (increase correlation for fraudulent transactions)
   - Negative correlation (blue in Class column): V1, V3, V5, V7, V10, V12, V14, V16, V17, V18 (decrease correlation for fraudulent transactions)
- 'Amount' shows weak correlation with 'Class', reiterate earlier observation that transaction amount alone is insufficient to detect fraud.
- 'Time' shows almost no correlation with 'Class', further supporting our decision to drop it during data preprocessing for model building.
- The features V1-V28 are anonymised PCA components, so no correlation interpretation can be made on these features to real problem.

### Data preprocessing
Based on EDA findings:
- 'Time' is dropped — no distinguishing pattern observed between valid and fraudulent transactions.
- 'Amount' is scaled using StandardScaler as it is the only non-PCA feature retained.
- V1–V28 are already PCA-transformed and do not require preprocessing.


### Scale & Prepare Features

In [ ]:
# Drop Time feature
df = df.drop(columns=['Time'])

# Scale Amount
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df[['Amount']])

# Separate features and target outcome
X = df.drop(columns=['Class'])
y = df['Class']

print(f"Features shape: {X.shape}")
print(f"Fraud cases: {y.sum()} ({y.sum() / len(y) * 100:.3f}%)")


### Split dataset into train/test ratio (80/20)
- Use stratify=y to control data split. This ensures the fraud/normal ratio is preserved in both train and test splits since dataset show severe class imbalance.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Fraud in test set: {y_test.sum()}")

### Model building for anomaly detection
- We used simple statistical thresholding-based flagging on 'Amount' and 'Frequency' of transactions, and the results were not able to show distinguishable pattern. In fact, it produced ambiguous results where the valid high-value transaction were flagged as false positive. Statistical methods rely on single feature and not accounting for the correlation between features.

- ML-based anomaly detection allow us to scale up the analysis complexity and use all features simultaneously. By doing this, we will be able to evaluate correlations and patterns that simple statistics fail to capture.

- We will use two ML models to do anomaly detection on the dataset:

**1) Isolation Forest (IF) model**
- Isolation Forest isolates anomalies by randomly selecting a feature, split on a random value between the feature's min and max, and repeat until an observation is isolated.
- The idea is that anomalies are few and different to normal transaction. So they requere fewer splits to isolate, resulting in shorter path lengths in the decision tree. As anomaly is calculated based on the path lenght, the shorter the path, the more likely the observation is an anomaly.

- We chose IF model because:
   - It uses 1 feature per split, no distance calculations needed, so it work efficiently with high-dimensional data (our dataset have 29 features!) with low computation cost.
   - It is scalable to large dataset such as ours with mare than 280K transactions.
   - It does not assume any undrelying data distribution, which is important given the heavily imbalanced dataset we had.
   
**2) Local Outlier Factor (LOF)**
- LOF is a density-based method that computes the local density deviation of each data point (transaction) relative to its neighbours. Points (transactions) in a significantly lower density regions than their neighbours are flagged as outliers.
- Interestingly, LOF inspect transactions within its own local cluster, enabling anomaly detection even if the whole data can appear normal. IF is not able to do this effectively.

- We chose LOF as a complementary model to IF because:
   - Its density-based approach is fundamentally different mechanisms to IF random splits mechanism.
   - It detects fraud/anomaly within local cluster, even when global clusetr look normal.

- LOF limitation: relies on distance metrics to define neighbourhoods, which can be less meaningful in high dimensional dataset. This is where IF model is expected to outperform LOF.
- The parameter 'n_neighbors' controls how many neighbours are considered. We use 'n_neighbours = 20', which is recommended as a general default.

*NOTE: We train models without the 'Class' (target outcome) label. We then evaluate models performance against the groudn truth labels to benchmark how well models detect fraudulent transactions.*

   

### Isolation Forest model


In [ ]:
# Fit Isolation Forest on training data
# No assumption made about the proportion of anomalies in the dataset - we won't know how many fraudsters in real fresh dataset so set 'contamination = auto'
# NOTE: initial modeling with 'contamination = auto' resulted in many False positives. Knowing the expected fraud rates improves model performance, so we set
# contamination to the known rate in the dataset.

# Contamination rate 
contamination = y.sum() / len(y)
print(f"Contamination: {contamination:.4f}")

iso_forest = IsolationForest(n_estimators=100,
                             # contamination='auto',   # contamination set to 'auto' reflecting unsupervised model, as one wouldn't know the true fraud rate upfront. 
                             random_state=RANDOM_SEED) 

iso_forest.fit(X_train)

# Predict on test data
y_pred_iso = iso_forest.predict(X_test)

# Isolation Forest return binary prediction 1 = normal, -1 = anomaly
# Map to match ground truth: 0 = valid, 1 = fraud
y_pred_iso = (y_pred_iso == -1).astype(int) 

In [ ]:
print("Predicted fraud cases:", y_pred_iso.sum())
print("Actual fraud cases:", y_test.sum())
print(f"Correct fraud detections:", ((y_pred_iso == 1) & (y_test.values == 1)).sum())

### Isolation Forest evaluation

In [ ]:
#Convert y_test to np.array to align with model prediction output
print("Isolation Forest classification report:")
print(classification_report(y_test.values, y_pred_iso, target_names=['Valid', 'Fraud']))

In [ ]:
# ROC_AUC Score
iso_score = iso_forest.decision_function(X_test)
ROC_AUC = roc_auc_score(y_test.values, -iso_score)
print(f"ROC_AUC score: {ROC_AUC:.2f}")

# Plot ROC-AUC curve
fpr, tpr, _ = roc_curve(y_test.values, -iso_score)

fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(fpr, tpr, color = 'steelblue',
        label = f'Isolation Forest (AUC = {ROC_AUC:.2f})')
ax.plot([0, 1], [0, 1], 'k--', 
        label='Random Classifier (AUC = 0.50)')
ax.set_title('Isolation Forest ROC AUC Curve')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')

plt.show()

In [ ]:
# Precision-Recall Curve
from sklearn.metrics import precision_recall_curve, auc

precision, recall, _ = precision_recall_curve(y_test.values, -iso_score)
pr_auc = auc(recall, precision)
fraud_rate = y_test.sum() / len(y_test)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall, precision, color='steelblue',
        label=f'Isolation Forest (AUC = {pr_auc:.2f})')
ax.axhline(y=fraud_rate, color='k', linestyle='--',
           label=f'Random Classifier (AUC = {fraud_rate:.4f})')
ax.set_title('Isolation Forest Precision Recall Curve')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(loc='upper right')
plt.show()


In [ ]:
# Confusion matrix
cm_iso = confusion_matrix(y_test.values, y_pred_iso)
ConfusionMatrixDisplay(confusion_matrix=cm_iso,
                       display_labels=['Valid', 'Fraud']).plot(cmap='viridis')
plt.title('Confusion Matrix Isolation Forest')
plt.show()

**Isolation Forest Results**

| Metric | Valid | Fraud |
|--------|-------|-------|
| Precision | 1.00 | 0.04 |
| Recall | 0.96 | 0.84 |
| F1 Score | 0.98 | 0.07 |

- ROC-AUC: 0.95

**Confusion Matrix**
| | Predicted Valid | Predicted Fraud |
|--------|-------|-------|
| **Actual Valid** | 54,811 (TN) | 2,053 (FP) |
| **Actual Fraud** | 16 (FN) | 82 (TP) |

*Observations*
- The model correctly identified **82 out of 98 fraud cases** (recall = 0.84) - strong performance for an unsupervised model with no fraud labels during training.
- **16 fraud cases were missed** (false negatives) — these are the most costly errors in fraud detection.
- **2,053 valid transactions were incorrectly flagged** as fraud (false positives) — resulting in very low precision of 0.04.
- **ROC-AUC of 0.95** indicates excellent overall discrimination ability between valid and fraudulent transactions but is partly affected by the large number of TN (54,811 valid transactions classified correctly).
- **PR-AUC = 0.17** tells a better story. When the model flags fraud, precision drops significantly as recall increases. The model struggles to maintain high precision while catching more fraud cases.
- The low precision reflects the fundamental challenge of anomaly detection on heavily imbalanced data. Model try to catch as many possible fraud cases, inevitably flagging many valid transactions in the process.
- The large gap between ROC-AUC and PR-AUC is characteristic of heavily imbalanced datasets — **PR-AUC is the more reliable metric**.
- Random classifier baseline for PR curve is just 0.0017 (the fraud rate), so PR-AUC of 0.17 is still 100x better than random, but far from perfect.
- This highlights the fundamental challenge of fraud detection, achieving high recall without sacrificing precision is difficult when fraud cases represent only 0.17% of transactions.


### Local Outlier Factor (LOF)



In [ ]:
# Fit LOF on test data
# LOF fit_predict in one step, no separate step to for model fitting then prediction
# Contamination set to known fraud rate for fair comparison with IF

contamination = y.sum() / len(y)

lof = LocalOutlierFactor(n_neighbors=20,
                         contamination=contamination)

y_pred_lof = lof.fit_predict(X_test)

# LOF binary prediction 1 = normal, -1 = anomaly
# Map to match ground truth: 0 = valid, 1 = fraud
y_pred_lof = (y_pred_lof == -1).astype(int) 

In [ ]:
print(f"Predicted fraud cases: {y_pred_lof.sum()}")
print(f"Actual fraud cases: {y_test.sum()}")
print(f"Correct fraud detections: {((y_pred_lof == 1) & (y_test.values == 1)).sum()}")

### LOF evaluation

In [ ]:
print("LOF Classification Report:")
print(classification_report(y_test.values, y_pred_lof, target_names=['Valid', 'Fraud']))

In [ ]:
# LOF anomaly score
lof_score = lof.negative_outlier_factor_

# ROC-AUC
roc_auc_lof = roc_auc_score(y_test.values, -lof_score)
print(f"ROC-AUC score: {roc_auc_lof:.2f}")

# Plot ROC-AUC curve
fpr_lof, tpr_lof, _ = roc_curve(y_test.values, -lof_score)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_lof, tpr_lof, color='crimson',
        label=f'LOF (AUC = {roc_auc_lof:.2f})')
ax.plot([0, 1], [0, 1], 'k--',
        label='Random Classifier (AUC = 0.50)')
ax.set_title('Local Outlier Factor ROC AUC Curve')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
plt.show()

In [ ]:
# LOF PR curve
precision_lof, recall_lof, _ = precision_recall_curve(y_test.values, -lof_score)
pr_auc_lof = auc(recall_lof, precision_lof)
print(f"PR-AUC Score: {pr_auc_lof:.4f}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall_lof, precision_lof, color='crimson',
        label=f'LOF (AUC = {pr_auc_lof:.4f})')
ax.axhline(y=fraud_rate, color='k', linestyle ='--',
        label=f'Random Classifier (AUC = {fraud_rate:.4f})')
ax.set_title('Local Outlier Factor Precision Recall Curve')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(loc='upper right')
plt.show()

In [ ]:
# Confusion matrix
cm_lof = confusion_matrix(y_test.values, y_pred_lof)
ConfusionMatrixDisplay(confusion_matrix=cm_lof,
                       display_labels=['Valid', 'Fraud']).plot(cmap='coolwarm')
plt.title('Local Outlier Factor Confusion Matrix')
plt.show()

**Local Outlier Factor Results**

| Metric | Valid | Fraud |
|--------|-------|-------|
| Precision | 1.00 | 0.00 |
| Recall | 1.00 | 0.00 |
| F1 Score | 1.00 | 0.00 |

**ROC-AUC: 0.51 | PR-AUC: 0.0022**

*Observations*
- LOF failed to detect a single fraud case, 0 out of 98 fraud transactions correctly identified.
- Despite predicting 99 anomalies, none overlapped with actual fraud cases. The model flagged the wrong transactions entirely.
- **ROC-AUC of 0.51** confirms LOF performs at near-random level on this dataset.
- **PR-AUC of 0.0022** is only marginally better than the random classifier baseline of 0.0017, effectively no discriminative power.
- This result validates our earlier concern about LOF in high dimensional space (29 features), distance metrics degrade and local density becomes 
  difficult to compute meaningfully.
- LOF appears to be flagging high-value valid transaction outliers rather than fraudulent transactions, consistent with what observed in the scatterplot EDA.
- **Isolation Forest significantly outperforms LOF** on this dataset, confirming that random split-based methods are more robust than distance-based methods for high dimensional fraud detection.

*The confusion matrix confirmed the observations*
| | Predicted Valid | Predicted Fraud |
|--------|-------|-------|
| **Actual Valid** | 56,765 (TN) | 99 (FP) |
| **Actual Fraud** | 98 (FN) | 0 (TP) |

   - 98 fraud transactions were missed, and every single transactions was classified as valid.
   - 99 valid transactions were incorrectly flagged as fraud. **THis is a complet failure in fraud detection**
   - 0 correct fraud detections.

  

### Model Comparison: Isolation Forest vs Local Outlier Factor

| Metric | Isolation Forest | Local Outlier Factor |
|--------|-----------------|---------------------|
| Predicted Fraud | 100 | 99 |
| Correct Detections (TP) | 82 | 0 |
| Missed Fraud (FN) | 16 | 98 |
| False Positives (FP) | 2,053 | 99 |
| Precision (Fraud) | 0.04 | 0.00 |
| Recall (Fraud) | 0.84 | 0.00 |
| F1 Score (Fraud) | 0.07 | 0.00 |
| ROC-AUC | 0.95 | 0.51 |
| PR-AUC | 0.17 | 0.0022 |

In [ ]:
# Overlay PR Curves
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(recall, precision, color='steelblue',
        label=f'Isolation Forest (AUC = {pr_auc:.2f})')
ax.plot(recall_lof, precision_lof, color='crimson',
        label=f'LOF (AUC = {pr_auc_lof:.4f})')
ax.axhline(y=fraud_rate, color='k', linestyle='--',
           label=f'Random Classifier (AUC = {fraud_rate:.4f})')
ax.set_title('Precision Recall Curve: Isolation Forest vs LOF')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(loc='upper right')
plt.show()

### Conclusion

We applied two unsupervised ML anomaly detection models to detect fraudulent credit card transactions in a heavily imbalanced dataset (0.172% fraud rate).

### Summary of Results

| Metric | Isolation Forest | Local Outlier Factor |
|--------|-----------------|---------------------|
| Correct Fraud Detections (TP) | 82 | 0 |
| Missed Fraud (FN) | 16 | 98 |
| False Positives (FP) | 2,053 | 99 |
| Precision (Fraud) | 0.04 | 0.00 |
| Recall (Fraud) | 0.84 | 0.00 |
| F1 Score (Fraud) | 0.07 | 0.00 |
| ROC-AUC | 0.95 | 0.51 |
| PR-AUC | 0.17 | 0.0022 |

### Key Findings

**Isolation Forest** performed significantly better than LOF:
- We correctly identified 82 out of 98 fraud cases (recall = 0.84) — strong performance for an unsupervised model that never saw fraud labels during training.
- ROC-AUC of 0.95 looks impressive but is partly inflated by the large number of correctly classified valid transactions.
- PR-AUC of 0.17 is 100x better than random (0.0017). This metric is the more honest reflection of fraud detection performance on this imbalanced dataset.
- Low precision (0.04) means many valid transactions were incorrectly flagged a known tradeoff when casting a wide net to catch fraud.

**Local Outlier Factor** failed to detect any fraud cases:
- 0 out of 98 fraud cases correctly identified. Complete failure for fraud detection.
- ROC-AUC of 0.51 and PR-AUC of 0.00 confirm near-random performance.
- Distance-based density metrics degraded in the 29-dimensional feature space, confirming our earlier concern about LOF in high dimensional data.

### What We Learned
- **High dimensionality matters** — IF's random split approach significantly outperforms LOF's distance-based approach in 29-dimensional space.
- **ROC-AUC alone is misleading** - for imbalanced datasets, always evaluate with PR-AUC for a more honest picture.
- **Unsupervised models can be powerful** — IF detected 84% of fraud without ever seeing a fraud label during training.
- **Precision-recall tradeoff is real** — catching more fraud inevitably means flagging more valid transactions as suspicious

### Further Work
- Tune IF 'n_estimators' and 'contamination' to improve precision.
- Explore deep learning approaches such as Autoencoders for anomaly detection.
- Combine IF with a supervised classifier in an ensemble to improve precision while maintaining high recall.